# HomeTown AR ml-service — interactive walkthrough

Runs the same three stages as `demo_run.py`/`run_service.py`, one cell at a time, so you can look at each
result before moving on and try different photos/prompts without re-running everything.

**Before this notebook will work**: run `setup_and_run.ps1` (or the manual steps in README.md) once from a
terminal to create `.venv` and fetch `vendor/TripoSR`. Then pick `.venv`'s Python as this notebook's kernel.

Each stage loads its model(s) fresh and frees GPU memory when the cell finishes — on an 8GB card, run cells
in order, top to bottom, not out of order, so you never have two stages' models resident at once.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # run from notebooks/, import the ml-service package from its parent

from pipeline import common

print("device:", common.device())
import torch
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"free VRAM: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

## Stage 1 — product photo -> 3D mesh
SAM2 segments the product, TripoSR reconstructs a mesh, the result is cleaned (largest shell kept,
normals fixed) and exported as a GLB. Swap `FURNITURE_PHOTO_URL` for a real catalogue photo
(`src/data/catalogue.json` -> `images[0]`) when you're ready to try a real product.

In [ ]:
import uuid
from pipeline.stage1_furniture3d import run_furniture3d

FURNITURE_PHOTO_URL = "https://images.unsplash.com/photo-1555041469-a586c61ea9bc?w=1024"

job = common.Job(id=str(uuid.uuid4()), kind="furniture3d", payload={"image_url": FURNITURE_PHOTO_URL})
result = run_furniture3d(job, progress=lambda p: print(f"\r{p:3d}%", end=""))
common.free_gpu()
print()
result

In [ ]:
# quick look at the mesh without leaving Python
import trimesh
mesh = trimesh.load(result["glb_path"])
mesh.show()  # opens an interactive viewer window; use mesh.bounds / mesh.extents to inspect instead if headless

## Stage 2 — room photo -> depth + detected furniture/architecture
Depth Anything V2 (relative depth, not metric) + Grounding DINO (open-vocabulary detection) + SAM2
(per-object masks). Advisory only — see the `note` field in the result.

In [ ]:
from pipeline.stage2_room_understanding import run_room_understanding

ROOM_PHOTO_URL = "https://images.unsplash.com/photo-1484154218962-a197022b5858?w=1024"

room_job = common.Job(id=str(uuid.uuid4()), kind="room_understanding", payload={"image_url": ROOM_PHOTO_URL})
room_result = run_room_understanding(room_job, progress=lambda p: print(f"\r{p:3d}%", end=""))
common.free_gpu()
print()
room_result

In [ ]:
from PIL import Image
Image.open(common.OUTPUT_DIR / f"{room_job.id}_depth.png")

## Stage 3 — room photo + instruction -> styled concept image
SDXL + a depth ControlNet (conditioned on stage 2's depth map, recomputed here) restyles the room
to the prompt while keeping its rough layout. **Design preview only** — never geometry.

In [ ]:
from pipeline.stage3_redesign import run_redesign

redesign_job = common.Job(
    id=str(uuid.uuid4()),
    kind="redesign",
    payload={"image_url": ROOM_PHOTO_URL, "prompt": "make this room minimalist and keep the existing sofa"},
)
redesign_result = run_redesign(redesign_job, progress=lambda p: print(f"\r{p:3d}%", end=""))
common.free_gpu()
print()
redesign_result

In [ ]:
from PIL import Image
Image.open(common.OUTPUT_DIR / f"{redesign_job.id}.png")